# Quant Factor Mining Agent — Brev Launchable Setup

[![Click here to deploy.](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/)

This launchable introduces NVIDIA's **Quant Factor Mining Agent** developer blueprint. It uses the [NeMo Agent Toolkit](https://docs.nvidia.com/nemo-agent-toolkit/) to orchestrate three LLM-powered agents that automatically generate, code, and evaluate alpha factors for quantitative finance.

In particular, the main notebook will cover the following:
- **Factor Mining Setup** – Clone the repository, install dependencies, and configure the environment
- **Factor Generation** – Use an LLM to propose factor expressions over price/volume operators
- **Code Generation** – Translate factor formulas into self-contained, executable Python
- **Backtesting on S&P 500 Data** – Compute Rank IC and statistical significance
- **Closed-Loop Optimization** – Iterate with advisor-driven feedback until the factor meets the IC threshold
- **Tracing with Phoenix** – Inspect every LLM call, token count, and latency end-to-end

In this **setup notebook**, we focus on the essential steps required to prepare the environment and install the dependencies. Once setup is complete, you will jump to the main notebook, [factor-mining-workflow.ipynb](../notebooks/factor-mining-workflow.ipynb).

---

<a id="environment-setup"></a>
## Environment Setup

Run the cells below to clone the repo, configure the project path, install dependencies, and register the Jupyter kernel.

**Note:** only clone the GitHub repository when deploying the Brev launchable. If you are running this notebook from inside an already-cloned checkout, skip the `git clone` cell.

In [ ]:
!git clone https://github.com/NVIDIA-AI-Blueprints/quant-factor-mining-agent.git

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()

candidates = [
    notebook_dir / "quant-factor-mining-agent",
    notebook_dir.parent,
    notebook_dir,
]
project_root = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the quant-factor-mining-agent checkout. "
        "Run the `git clone` cell above first, or open this notebook from inside the repo."
    )

local_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = f"{local_bin}:{os.environ['PATH']}"
print(f"Updated PATH to include: {local_bin}")

sys.path.insert(0, str(project_root / "src"))
os.chdir(project_root)
print(f"Working directory changed to: {Path.cwd()}")

Updated PATH to include: /Users/phuo/.local/bin
Working directory changed to: /Users/phuo/Desktop/quant-factor-mining-agent


### Install `uv` and Project Dependencies

The repo is managed with [`uv`](https://docs.astral.sh/uv/), a fast Python package manager. The cell below installs `uv` if missing, creates the project virtual environment, and registers a dedicated Jupyter kernel named **Factor Mining**.

In [3]:
import subprocess

subprocess.run("curl -sSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)

install_script = """
unset VIRTUAL_ENV
uv venv
uv pip install -e ".[test]"
uv run python -m ipykernel install --user --name=factor-mining --display-name "Factor Mining"
"""

subprocess.run(install_script, shell=True, executable="/bin/bash", check=True)
print("\n\u2705 Environment ready and 'Factor Mining' kernel registered.")

Using CPython 3.12.12 interpreter at: /opt/homebrew/opt/python@3.12/bin/python3.12
Creating virtual environment at: .venv
error: Failed to create virtual environment
  Caused by: A virtual environment already exists at `/Users/phuo/Desktop/quant-factor-mining-agent/.venv`. Use `--clear` to replace it
Resolved 270 packages in 928ms
   Building factor-mining-workflow @ file:///Users/phuo/Desktop/quant-factor-mining-agent
      Built factor-mining-workflow @ file:///Users/phuo/Desktop/quant-factor-mining-agent
Prepared 1 package in 346ms
Uninstalled 1 package in 1ms
Installed 1 package in 3ms
 ~ factor-mining-workflow==0.0.0 (from file:///Users/phuo/Desktop/quant-factor-mining-agent)


Installed kernelspec factor-mining in /Users/phuo/Library/Jupyter/kernels/factor-mining

✅ Environment ready and 'Factor Mining' kernel registered.


### Configure Your NVIDIA API Key

The workflow calls hosted NVIDIA NIM endpoints for the Factor, Code, and Advisor agents. Get a key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys) and paste it below.

The key is written to a `.env` file at the project root so the main notebook can pick it up automatically; it is **not** committed to git (the repo's `.gitignore` covers it).

In [4]:
import getpass
from pathlib import Path

env_path = Path(".env")

def _load_env_file(path: Path) -> dict[str, str]:
    """Parse a simple KEY=VALUE .env file. Returns {} if the file is missing."""
    if not path.exists():
        return {}
    out = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        out[k.strip()] = v.strip().strip('"').strip("'")
    return out

env_file = _load_env_file(env_path)

api_key = (
    os.environ.get("NVIDIA_API_KEY")
    or env_file.get("NVIDIA_API_KEY")
    or getpass.getpass("Enter your NVIDIA_API_KEY: ").strip()
)
if not api_key:
    raise ValueError("NVIDIA_API_KEY is required to run the workflow.")

os.environ["NVIDIA_API_KEY"] = api_key

env_file["NVIDIA_API_KEY"] = api_key
env_path.write_text("\n".join(f"{k}={v}" for k, v in env_file.items()) + "\n")
print(f"\u2705 NVIDIA_API_KEY set in environment and persisted to {env_path.resolve()}")
print("   (.env is gitignored — safe to keep on disk.)")

✅ NVIDIA_API_KEY set in environment and persisted to /Users/phuo/Desktop/quant-factor-mining-agent/.env
   (.env is gitignored — safe to keep on disk.)


### Download S&P 500 Price/Volume Data

The workflow backtests against daily OHLCV data fetched via [`yfinance`](https://github.com/ranaroussi/yfinance). The cell below downloads the dataset only if it is not already present.

In [5]:
data_dir = Path("src/factor_mining_workflow/data/sp500")
if data_dir.exists() and any(data_dir.glob("*.csv")):
    print(f"\u2705 Data already present at {data_dir} — skipping download.")
else:
    subprocess.run(
        "uv run python -m factor_mining_workflow.download_data",
        shell=True, executable="/bin/bash", check=True,
    )
    print("\u2705 Data downloaded.")

✅ Data already present at src/factor_mining_workflow/data/sp500 — skipping download.


### Verify Installation

After the install cell finishes, **select the `Factor Mining` kernel** from the kernel menu (top-right of Jupyter), then run the cell below to confirm every required package imports cleanly.

In [6]:
import importlib

packages = [
    "numpy", "pandas", "scipy", "yfinance",
    "nat", "phoenix",
    "factor_mining_workflow",
]

print("Checking packages...\n")
failed = []
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"\u2713 {pkg:30s} {ver}")
    except ImportError:
        print(f"\u2717 {pkg:30s} FAILED")
        failed.append(pkg)

summary = ('\u274c Missing: ' + ', '.join(failed)) if failed else '\u2705 All packages ready!'
print(f"\n{summary}")

Checking packages...

✓ numpy                          2.4.2
✓ pandas                         2.3.3
✓ scipy                          1.17.0
✓ yfinance                       1.2.0
✓ nat                            installed
✓ phoenix                        13.11.0
✓ factor_mining_workflow         installed

✅ All packages ready!


---
## Setup Complete!

If you see **✅ All packages ready!** above, the Python environment, kernel, API key, and dataset are all in place.

You can now jump to the main walkthrough:

➡️ **[Open the Factor Mining Workflow Notebook](../notebooks/factor-mining-workflow.ipynb)** ⬅️

Make sure to switch the kernel of that notebook to **Factor Mining** before running its cells.

---

SPDX-FileCopyrightText: Copyright (c) 2023-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0. Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.